# Wine 데이터 딥러닝 분류 실습

수업에서 받은 diabetes 딥러닝 예제를 Wine 데이터에 적용해 보겠습니다.

diabetes는 0/1을 맞히는 **2클래스** 문제이고, Wine은 와인 품종 0/1/2를 맞히는 **3클래스** 문제입니다.
그래서 데이터를 불러오는 부분, 정답의 원-핫 인코딩, 출력층, loss, 예측 부분이 바뀝니다. 바뀐 줄에는 `# [변경]` 주석을 달았습니다.

| 구분 | diabetes | Wine |
|---|---|---|
| 입력 특성 | 8개 | 13개 |
| 정답 | Outcome (0/1) | target (0/1/2) |
| 정답 형태 | 0/1 그대로 | 원-핫 (예: 1 → [0, 1, 0]) |
| 데이터 수 | 768개 | 178개 |
| 출력층 | Dense(1, sigmoid) | Dense(3, softmax) |
| loss | binary_crossentropy | categorical_crossentropy |

1. 데이터 불러오기
2. 입력 / 정답 분리
3. 학습 / 테스트 데이터 분리
4. 데이터 정규화
5. 모델 생성
6. 모델 설정
7. 모델 구조 출력
8. 모델 학습
9. 테스트 데이터 예측
10. 성능 평가

## 0. 라이브러리 불러오기

Wine 데이터는 `scikit-learn`에 들어 있어서 파일 없이 `load_wine`으로 불러옵니다.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.utils import to_categorical          # [변경] 원-핫 인코딩용
from sklearn.datasets import load_wine                     # [변경] Wine 데이터 불러오기용
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. 데이터 불러오기

`wine.frame`에는 13개 화학 성분 열과 정답인 `target` 열이 함께 들어 있습니다.

In [2]:
wine = load_wine(as_frame=True)                            # [변경] read_csv 대신 load_wine
df = wine.frame

print(df.head())
print(df.shape)

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  target  
0          

## 2. 입력 / 정답 분리

정답 열 이름이 `Outcome`에서 `target`으로 바뀝니다.

In [3]:
X = df.drop("target", axis=1).values                       # [변경] "Outcome" → "target"
y = df["target"].values                                    # [변경] "Outcome" → "target"

print("X shape:", X.shape)
print("클래스별 개수:", np.bincount(y))

X shape: (178, 13)
클래스별 개수: [59 71 48]


## 3. 학습 / 테스트 데이터 분리

예제와 같이 20%를 테스트용으로 떼어 둡니다. `stratify=y`로 세 클래스 비율을 학습/테스트에 똑같이 유지합니다.

나눈 뒤에 학습용 정답 `y_train`을 **원-핫 인코딩**합니다.

| 정수 라벨 | 원-핫 |
|---|---|
| 0 | [1, 0, 0] |
| 1 | [0, 1, 0] |
| 2 | [0, 0, 1] |

원-핫 인코딩을 사용하는 이유

- 출력층 softmax는 `[0.1, 0.8, 0.1]`처럼 클래스마다 확률을 하나씩 냅니다. 정답도 같은 모양 `[0, 1, 0]`이어야 클래스별로 비교해서 loss를 계산할 수 있습니다.
- 정수 0, 1, 2를 그대로 쓰면 "2가 1보다 크다"는 크기·순서 관계가 있는 것처럼 보이지만, 와인 품종 사이에는 그런 관계가 없습니다. 원-핫은 모든 클래스를 동등하게 표현합니다.

`y_test`는 정수 그대로 둡니다. 10번에서 `argmax`로 구한 예측 클래스 번호(정수)와 바로 비교하기 위해서입니다.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_train = to_categorical(y_train)                          # [변경] 정답 원-핫 인코딩

print("학습:", X_train.shape, "테스트:", X_test.shape)
print("원-핫 정답 예시:", y_train[0])

학습: (142, 13) 테스트: (36, 13)
원-핫 정답 예시: [1. 0. 0.]


## 4. 데이터 정규화

Wine 특성은 값의 크기가 많이 다릅니다(예: proline은 수백~수천, hue는 1 안팎). 그래서 평균 0, 표준편차 1로 맞춰 줍니다.

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 5. 모델 생성

- 입력 특성이 13개라서 `input_shape=(13,)`
- 클래스가 3개라서 출력층은 `Dense(3, activation="softmax")`. 세 클래스의 확률을 내고, 그 합은 1입니다.

In [6]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(32, activation="relu", input_shape=(13,)))  # [변경] 8 → 13
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(3, activation="softmax"))                   # [변경] 1, sigmoid → 3, softmax

C:\Users\ektlaksskgodqhrgo\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\layers\core\dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## 6. 모델 설정

정답을 원-핫으로 바꿨으므로 다중 분류용 `categorical_crossentropy`를 사용합니다.

예제에서는 `compile`이 5번과 6번에 두 번 있었는데, 여기서는 한 번만 합니다.

In [7]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])  # [변경] binary → categorical

## 7. 모델 구조 출력

In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027 (4.01 KB)

 Trainable params: 1,027 (4.01 KB)

 Non-trainable params: 0 (0.00 B)

## 8. 모델 학습

예제와 같은 설정으로 학습합니다. 학습 데이터 142개 중 20%(약 29개)는 검증용으로 사용됩니다.

In [9]:
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 1s 664ms/step - accuracy: 0.1875 - loss: 1.1806

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.3009 - loss: 1.1233

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 171ms/step - accuracy: 0.3009 - loss: 1.1233 - val_accuracy: 0.4828 - val_loss: 0.9298


Epoch 2/100


...

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.0060

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0084 - val_accuracy: 1.0000 - val_loss: 0.0117


Epoch 100/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.0059

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.0082 - val_accuracy: 1.0000 - val_loss: 0.0115


## 9. 테스트 데이터 예측

모델은 샘플마다 `[0.1, 0.8, 0.1]`처럼 세 클래스의 확률을 냅니다.
`argmax`로 가장 큰 확률의 위치(클래스 번호)를 예측값으로 씁니다.

In [10]:
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)                         # [변경] >= 0.5 대신 argmax

print("확률 예시:", y_prob[0].round(3), "→ 예측:", y_pred[0], "/ 정답:", y_test[0])

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


확률 예시: [1. 0. 0.] → 예측: 0 / 정답: 0


## 10. 성능 평가

혼동 행렬은 3×3으로 나옵니다. 행은 실제 클래스, 열은 예측 클래스입니다.

In [11]:
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy =", accuracy)
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))


Accuracy = 0.9722222222222222

Confusion Matrix
[[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      1.00      0.97        14
           2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

